# Per-Frame Possession Ground Truth: Labelling Tool

**Purpose:** An interactive labelling tool: a human records the ball holder for every
frame of all three clips (534 frames: clip_1 117, clip_2 174, clip_3 243), judged
frame by frame.  
**Inputs:** `data/raw/clip_*.mp4`, cached player tracks and gated ball detections in
`data/processed/`.  
**Outputs:** `data/annotations/possession_gt_per_frame.csv`, appended and fsynced
after every answer, resumable.  
**Backs:** the shipped possession ground truth in `data/annotations/`, which this tool
produced; the possession sweep and event scoring are scored against it.

Frames are presented chronologically within each clip, clips in fixed `clip_1,
clip_2, clip_3` order, with no sampling and no shuffling. Chronological order is
deliberate and is the opposite of the team GT notebook's fixed shuffle. There,
shuffling stopped a labeller propagating one remembered team across a track's whole
life through an ID switch. Possession is instead a temporal reading of the play: who
is holding the ball is judged from the possession developing across neighbouring
frames, so presenting frames out of order would remove the context the answer depends
on.

Only real, gated ball detections are ever drawn. Frames the detector never saw the
ball on are captioned as such rather than being shown an interpolated box: a
fabricated bbox would lead the labeller to a fabricated answer, contaminating the
ground truth with exactly the interpolation the possession sweep is meant to be scored
against.

Frames are decoded one clip at a time, not all three up front. All 534 frames held
simultaneously would be several GB resident; `make_clip_loader()` keeps at most one
clip's frames in memory, safe because labelling proceeds strictly clip by clip, so an
earlier clip is never requested again once labelling has moved past it.

The UI is matplotlib inline plus `input()`, not ipywidgets, because
`jupyterlab_widgets` is not registered with this project's JupyterHub server process,
so widgets do not render there.

All logic lives in `basketball/labelling/possession_gt.py` and
`basketball/labelling/possession_rendering.py` and is unit-tested; this notebook only
supplies the display and prompt callables.

## Setup

In [ ]:
import os
from pathlib import Path

# Every path below is relative to the repo root, not wherever Jupyter
# happened to start.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
os.chdir(REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from basketball.labelling.possession_gt import (
    CLIPS,
    CSV_PATH,
    build_frame_order,
    default_frame_counts,
    frame_player_ids_for_all_clips,
    load_existing_labels,
    make_clip_loader,
    run_session,
)
from basketball.labelling.possession_rendering import render_labelling_frame, to_rgb

## Per-clip lazy loader

Creates `clip_loader`, which decodes ONE clip's frames (plus its cached
tracks and gated ball detections) on first request and discards whatever
clip was cached before whenever a DIFFERENT clip is asked for -- see the
title cell above for why this is safe. Nothing is decoded yet; the first
'loaded N frames' line below appears once the Label cell further down
actually reaches clip_1's first frame.

In [ ]:
clip_loader = make_clip_loader()
print(
    'Lazy per-clip loader ready -- frames decode on demand, one clip at a '
    'time. Watch for a "loaded N frames (M MB)" line the first time each '
    'clip below is reached.'
)

## Build the chronological order and show progress

In [ ]:
frame_counts = default_frame_counts()
frame_order = build_frame_order(frame_counts)
frame_player_ids = frame_player_ids_for_all_clips()

# Progress is read through load_existing_labels(), which dedups (clip,
# frame_idx) last-wins, so a corrected frame counts once rather than twice.
already_labelled = load_existing_labels(CSV_PATH)
print(f'Frames to label: {len(frame_order)} ({frame_counts})')
print(f'Already labelled: {len(already_labelled)}')
print(f'Remaining: {len(frame_order) - len(already_labelled)}')

## Label

One image per frame: every tracked player box with its `track_id`, plus the
gated ball box in white (captioned `BALL (detected)`) when the detector saw
the ball, or a `NO BALL DETECTION THIS FRAME` caption when it did not.

Answer with the holder's **track id**, or `n` (nobody holds the ball), `u`
(unclear), `s` (stop and save). Anything else, or a track id not visible in
that frame, is rejected and the same frame is presented again without
writing anything.

Every answer is appended and fsynced immediately, so a kernel death loses at
most the frame in progress. Re-running this cell resumes at the first
unlabelled frame, recomputed by scanning the full 534-frame list.

In [ ]:
def show(clip: str, frame_idx: int) -> None:
    """Display the annotated frame for one labelling decision, inline; loads (and discards any other clip's data) via clip_loader on demand."""
    data = clip_loader(clip)
    annotated = render_labelling_frame(data.frames[frame_idx], data.tracks[frame_idx], data.gated[frame_idx])
    plt.figure(figsize=(14, 8))
    plt.imshow(to_rgb(annotated))
    plt.axis('off')
    plt.title(f'{clip} - frame {frame_idx}')
    plt.show()


written = run_session(
    frame_order=frame_order,
    frame_player_ids=frame_player_ids,
    show=show,
    prompt=input,
    path=CSV_PATH,
)
print(f'{written} label(s) written to {CSV_PATH}')

## Verify what was written

In [ ]:
from collections import Counter

from basketball.labelling.possession_gt import load_labelled_rows

rows = load_labelled_rows(CSV_PATH)
holders = Counter(row['holder'] for row in rows)
print(f'{len(rows)} deduplicated label(s) on disk')
print(f'nobody: {holders.get("nobody", 0)}, unclear: {holders.get("unclear", 0)}, '
      f'named holders: {sum(v for k, v in holders.items() if k not in ("nobody", "unclear"))}')
for clip in CLIPS:
    labelled = sum(1 for row in rows if row['clip'] == clip)
    print(f'{clip}: {labelled}/{frame_counts[clip]} frames labelled')